# Notebook 02 - Supports, Boundary Conditions, and Load Cases

This lesson adds the structural context that turns geometry into a solvable engineering model. It is still a setup notebook: you define supports and load cases here, then Notebook 03 runs/imports Code_Aster results.

You will do four things:

1. Compare anchor, guide, rest, spring, and custom supports.
2. Attach directional stiffness, mass, and friction metadata.
3. Visualize support hardware in context.
4. Define hot, cold, and hydrotest load cases.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pyvista as pv
from tuba import Model, Material, PipeSection, PipingBuilder
from tuba.visualizer import plots
from tuba.visualizer.pipeline import build_mesh_from_model, inflate_tubes

# Defaults to zoomable embedded HTML locally; set TUBA_NOTEBOOK_BACKEND=client or static to override.
from tuba.visualizer.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()
print("Imports ready.")

## 1. Support Types Overview

Tuba v4 provides five built-in support types. Each maps to a distinct structural behavior and renders as a distinct 3D support shape.

| Type | Structural behavior | Typical use |
|---|---|---|
| **anchor** | All six DOFs fixed | Equipment tie-ins and fixed endpoints |
| **guide** | Lateral restraint with axial sliding | Thermal growth control |
| **rest** | Vertical support with horizontal sliding | Pipe-rack shoe supports |
| **spring** | Elastic restraint with specified stiffness | Vertical flexibility or hangers |
| **custom** | User-defined blocked DOFs | Directional stops and non-standard restraints |

Choosing the right support type is critical for realistic stress results. Over-constraining a system can artificially raise thermal-expansion stress.


In [ ]:
# ── Build a model showcasing every support type ──────────────────────

model = Model("SupportDemo", standard="ASME_B31.3")

# Material: Carbon Steel
model.add_material(
    "Steel",
    E=2.1e11,       # Young's modulus [Pa]
    nu=0.3,         # Poisson's ratio
    rho=7850.0,     # Density [kg/m³]
    alpha=1.2e-5,   # Thermal expansion [1/°C]
    allowable_stress={20.0: 137e6, 100.0: 130e6, 200.0: 120e6},
)

# Pipe section: DN100 (4" Sch 40)
model.add_pipe_section("DN100", OD=0.1143, WT=0.00602, corrosion_allowance=0.001)

# ── Piping route: 10 m straight run along +X ────────────────────────
with model.pipe(section="DN100", material="Steel") as b:
    b.start([0, 0, 0], support="anchor")          # Node 0 — ANCHOR (red)
    b.run(2.0)
    b.add_support(type="guide")                    # @ 2 m  — GUIDE (green)
    b.run(2.0)
    b.add_support(type="rest")                     # @ 4 m  — REST (blue)
    b.run(2.0)
    b.spring(y=1.5e6)                                # @ 6 m  — Y-SPRING (yellow)
    b.run(2.0)
    b.add_support(type="custom",                   # @ 8 m  — CUSTOM (magenta)
                  blocked_dof=[1, 1, 0, 0, 0, 1])  # Tx, Ty, Rz blocked
    b.run(2.0)
    b.end(support="anchor")                        # Node end — ANCHOR (red)

# ── Inspect supports ────────────────────────────────────────────────
print(f"Model '{model.project_name}' — {len(model.supports)} supports\n")
for s in model.supports:
    print(f"  Node {s.node:>4s}  │  type={s.type:<8s}  │  props: "
          f"stiffness_matrix={getattr(s, 'stiffness_matrix', '—')}  "
          f"blocked_dof={getattr(s, 'blocked_dof', '—')}")

In [ ]:
# ── 3D Render: every support shape in context ───────────────────────

mesh = build_mesh_from_model(model)
tubes = inflate_tubes(mesh, radius=0.05)

p = pv.Plotter()
p.set_background("#1a1a2e")

# Add pipe tubes
p.add_mesh(tubes, color="#c0c0c0", smooth_shading=True, opacity=0.85)

# Add 3D support shapes (anchor blocks, guide collars, etc.)
plots._add_supports_to_plotter(p, model, scale=0.18)

# Legend
p.add_legend(
    [
        ["Anchor", "red"],
        ["Guide", "green"],
        ["Rest", "blue"],
        ["Spring", "yellow"],
        ["Custom", "magenta"],
    ],
    bcolor="#2a2a3e",
    face="circle",
)

p.camera_position = "xz"
p.show(jupyter_backend=JUPYTER_BACKEND)

## 2. Advanced Support Configuration

The overview model already shows the five support types. This section adds the data that matters to solver behavior: directional spring stiffness, concentrated mass, and friction.

### Spring Stiffness Matrix

A spring support can define translational and rotational stiffness values per DOF.

### Concentrated Mass

Mass on a support node can represent heavy valves or in-line equipment.

### Friction Coefficient

Friction on rest supports influences sliding resistance during thermal growth.


In [ ]:
# ── Advanced support examples ───────────────────────────────────────

# Directional spring: stiff vertically, softer laterally, free rotation
model.add_support(
    node="N3",
    type="spring",
    stiffness_matrix=[1e5, 2e5, 3e5, 0, 0, 0],
)

# Rest with concentrated mass (valve weight) and friction
model.add_support(
    node="N2",
    type="rest",
    mass=50.0,                  # 50 kg lumped mass
    friction_coefficient=0.3,   # Coulomb μ for steel-on-steel
)

# ── Print updated supports ──────────────────────────────────────────
print(f"Updated support count: {len(model.supports)}\n")
for s in model.supports:
    extras = []
    if hasattr(s, 'stiffness_matrix') and s.stiffness_matrix:
        extras.append(f"stiffness_matrix={s.stiffness_matrix}")
    if hasattr(s, 'mass') and s.mass:
        extras.append(f"mass={s.mass} kg")
    if hasattr(s, 'friction_coefficient') and s.friction_coefficient:
        extras.append(f"μ={s.friction_coefficient}")
    extra_str = ", ".join(extras) if extras else "—"
    print(f"  Node {s.node:>4s}  │  {s.type:<8s}  │  {extra_str}")

## 3. Load Cases

A load case bundles the actions applied to the piping system in one operating scenario.

| Parameter | Description |
|---|---|
| `gravity` | Self-weight and contents weight |
| `pressure` | Internal design pressure in Pa |
| `temperature` | Operating temperature in degrees C |
| `ref_temperature` | Installation or ambient temperature in degrees C |

Thermal expansion stress is driven by `temperature - ref_temperature`.


In [ ]:
# ── Define load cases ───────────────────────────────────────────────

model.define_load_case(
    "Operating_Hot",
    gravity=True,
    pressure=2.5e6,         # 25 bar
    temperature=220.0,      # °C
    ref_temperature=20.0,   # °C  →  ΔT = 200 °C
)

model.define_load_case(
    "Operating_Cold",
    gravity=True,
    pressure=0.5e6,         # 5 bar
    temperature=50.0,       # °C
)

model.define_load_case(
    "Hydrotest",
    gravity=True,
    pressure=3.75e6,        # 1.5 × 25 bar
    temperature=20.0,       # ambient
)

# ── Inspect ──────────────────────────────────────────────────────────
print(f"{len(model.load_cases)} load cases defined:\n")
for lc in model.load_cases.values():
    print(f"  📦 {lc.name:<16s}  │  gravity={str(lc.gravity):<5s}  "
          f"│  P={lc.internal_pressure/1e6:.2f} MPa  "
          f"│  T={lc.temperature:.0f} °C  "
          f"│  T_ref={getattr(lc, 'ref_temperature', 20.0):.0f} °C")

## Key Takeaways

| Support | Use it for |
|---|---|
| **Anchor** | Nozzle connections, equipment tie-ins, fixed points |
| **Guide** | Mid-span lateral restraint while allowing thermal growth |
| **Rest** | Gravity supports on pipe racks and shoe supports |
| **Spring** | Vertical flexibility with specified stiffness |
| **Custom** | Non-standard restraints and partial fixity |

- Use `stiffness_matrix` when directional stiffness differs by DOF.
- Attach `mass` to support nodes for heavy in-line components.
- Set `friction_coefficient` on rests when sliding resistance matters.
- Load cases define the solver envelope used by the Code_Aster and ASME B31.3 workflow in Notebook 03.

Next: `03_stress_analysis_and_compliance.ipynb` runs/imports Code_Aster results and checks compliance.
